In [ ]:
# NOTE: This notebook is used for computing spatial metrics, following deconvolution analysis.

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import pickle as pkl
import json
import os

import squidpy as sq
import scanpy as sc

from tqdm import tqdm

In [11]:
# Loading the processed data

adata_full = ad.read_h5ad("data_processed/adata_full.h5ad")
adata_malignant = ad.read_h5ad("data_processed/adata_malignant.h5ad")

with open("data_processed/aggregated_malignant_df_with_assignments.pkl", "rb") as file:
    aggregated_malignant_df = pkl.load(file)

/Users/alexstihi/.pyenv/versions/oxford_venv/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [12]:
aggregated_malignant_df.head()

,Patient_ID,Frame,Grid_X_Y,No_cells,Cell_names,APOA1,DCN,DPP4,ENG,IGFBP5,...,MYH11_raw,PIGR_raw,PTGS1_raw,RAMP1_raw,RGS5_raw,S100A4_raw,SPP1_raw,TPM2_raw,UBE2C_raw,Assignment
0,HGSC1,F001,0_0,19,"SMI_T10_F001_c7207,SMI_T10_F001_c7287,SMI_T10_...",1.573922,2.668908,0.934569,2.999990,1.109909,...,0.0,0.0,2.0,1.0,1.0,5.0,1.0,20.0,8.0,EMT
1,HGSC1,F001,0_1,14,"SMI_T10_F001_c6601,SMI_T10_F001_c6744,SMI_T10_...",0.589779,1.999994,0.999998,1.535002,0.635354,...,0.0,1.0,6.0,1.0,1.0,6.0,2.0,12.0,7.0,C10
2,HGSC1,F001,0_2,19,"SMI_T10_F001_c6459,SMI_T10_F001_c6106,SMI_T10_...",1.837428,1.999994,0.999998,0.000000,0.870693,...,0.0,2.0,1.0,2.0,2.0,3.0,4.0,18.0,5.0,EMT
3,HGSC1,F001,0_3,25,"SMI_T10_F001_c5654,SMI_T10_F001_c5766,SMI_T10_...",3.707808,0.000000,1.977611,0.000000,0.000000,...,0.0,1.0,6.0,2.0,4.0,4.0,1.0,13.0,10.0,C10
4,HGSC1,F001,0_4,23,"SMI_T10_F001_c5346,SMI_T10_F001_c5170,SMI_T10_...",1.729487,0.963116,1.999996,0.000000,1.401605,...,0.0,1.0,5.0,1.0,1.0,4.0,3.0,8.0,6.0,C10


In [13]:
# Feed back the assigned labels to the full anndata object

def process_adata_with_asignment(adata_full, aggregated_malignant_df):

    # copy to avoid in-place modifications
    ad = adata_full.copy()

    cell_to_assignment = {}

    for _, row in aggregated_malignant_df.iterrows():
        cell_names = row["Cell_names"].split(",")
        assignment = row["Assignment"]
        
        for cell_name in cell_names:
            cell_to_assignment[cell_name] = assignment
    # Change this to the obs column that currently holds labels with "Malignant"
    base_col = "Cell_type"  # e.g., "cell_type", "CellType", etc.

    # Map Cell_ID -> assignment (NaN if not present)
    assignments = ad.obs["Cell_ID"].map(cell_to_assignment)

    # New refined column
    refined = ad.obs[base_col].astype(str).copy()
    mask = refined == "Malignant"
    refined.loc[mask] = (
        ("Malignant." + assignments.loc[mask].astype(str))
        .where(assignments.loc[mask].notna(), "Malignant.Other")
    )

    ad.obs["Malignant_assignment"] = assignments.astype("category")
    ad.obs[f"{base_col}_refined"] = refined.astype("category")

    print(
        f"Malignant cells: {int(mask.sum())} | "
        f"mapped: {int(assignments.loc[mask].notna().sum())} | "
        f"unmapped: {int(mask.sum() - assignments.loc[mask].notna().sum())}"
    )

    # Replace the old "Cell_type" with refined values and drop the helper column
    ad.obs[base_col] = ad.obs[f"{base_col}_refined"].astype("category")
    ad.obs.drop(columns=[f"{base_col}_refined"], inplace=True)
    ad.obs.drop(columns=["Malignant_assignment"], inplace=True)

    # Add a separate frame column, called "Frame"
    ad.obs["Frame"] = ad.obs["Frame"].str.split("_").str[2]

    ad.obs.drop(columns=["Frame_no_cells", "Frame_x", "Frame_y", "Frame_size", "Frame_cell_dens"], inplace=True)

    # Create a new column for the spatial coordinates and drop them from obs
    # ad.obsm["spatial"] = ad.obs[["Local_x", "Local_y"]].values
    # ad.obs.drop(columns=["Local_x", "Local_y"], inplace=True)

    return ad

adata_spatial = process_adata_with_asignment(adata_full, aggregated_malignant_df)

Malignant cells: 286242 | mapped: 239825 | unmapped: 46417


In [14]:
adata_spatial.obs.head()

,Local_x,Local_y,Local_row,Cell_type,Patient,Frame,TMA,Cell_ID
0,1493.3500,3151.233,SMI_T10_F001_c1017,Malignant.C10,HGSC1,F001,T10,SMI_T10_F001_c1017
1,850.5670,3143.833,SMI_T10_F001_c1062,Malignant.C10,HGSC1,F001,T10,SMI_T10_F001_c1062
2,3222.7300,3152.586,SMI_T10_F001_c1064,Malignant.ciliated,HGSC1,F001,T10,SMI_T10_F001_c1064
3,78.0125,3179.012,SMI_T10_F001_c1075,Malignant.Other,HGSC1,F001,T10,SMI_T10_F001_c1075
4,1169.2200,3156.375,SMI_T10_F001_c1079,Malignant.C10,HGSC1,F001,T10,SMI_T10_F001_c1079


In [7]:
# Extract Patient-FOV combinations
patient_fovs = adata_spatial.obs.groupby(["Patient", "Frame"])

/var/folders/dh/0xrq7_y539x00hjtfjyrjds00000gn/T/ipykernel_6827/700657912.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  patient_fovs = adata_spatial.obs.groupby(["Patient", "Frame"])


In [43]:
# Compute relative proportions of Malignant cells (only Malignant.C3, Malignant.C4, Malignant.EMT, Malignant.C10, Malignant.ciliated)
# 1. Relative to the total malignant cells 
# 2. Relative to the total cells in the frame.


def compute_malignant_proportions(adata_spatial, patient_fovs):
    malignant_proportions_df= pd.DataFrame(columns=["Patient_ID", "Frame", "C3", "C4", "EMT", "C10", "ciliated", "Malignant (%)", "Total Cells"])

    i = 0
    for (patient, fov), indices in tqdm(patient_fovs.indices.items(), desc="Processing patients"):

        # print(f"Analyzing {patient} - FOV {fov}")

        # subset the data
        adata_subset = adata_spatial[indices].copy()

        # create empty DataFrame to store proportions
        proportions_df = pd.DataFrame(columns=["Patient_ID", "Frame", "C3", "C4", "EMT", "C10", "ciliated", "Malignant (%)", "Total Cells"])

        # count how many malignant cells are present
        malignant_mask = adata_subset.obs['Cell_type'].isin(['Malignant.C3', 'Malignant.C4', 'Malignant.EMT', 'Malignant.C10', 'Malignant.ciliated'])
        total_malignant_counts = malignant_mask.sum()

        # individual malignant proportions
        for cell_type in ['Malignant.C3', 'Malignant.C4', 'Malignant.EMT', 'Malignant.C10', 'Malignant.ciliated']:
            cell_type_mask = adata_subset.obs['Cell_type'] == cell_type
            cell_count = cell_type_mask.sum()
            proportion = cell_count / total_malignant_counts * 100 if total_malignant_counts > 0 else 0

            # store proportions  - concatenate
            proportions_df.loc[i, "Patient_ID"] = patient
            proportions_df.loc[i, "Frame"] = fov
            proportions_df.loc[i, cell_type.split(".")[-1]] = proportion

        # Compute the relative proportions of the above Malignant cells to all the cells in the frame - exclude Malignant.Other cells
        total_counts = adata_subset.obs['Cell_type'].value_counts().sum() - adata_subset.obs['Cell_type'].value_counts().get('Malignant.Other', 0)
        proportions_df.loc[i, "Total Cells"] = total_counts
        proportions_df.loc[i, "Malignant (%)"] = (total_malignant_counts / total_counts) * 100 if total_counts > 0 else 0

        # concatenate with malignant_proportions_df
        malignant_proportions_df = pd.concat([malignant_proportions_df, proportions_df], ignore_index=True)

        i += 1

    return malignant_proportions_df

malignant_proportions_df = compute_malignant_proportions(adata_spatial, patient_fovs)

Processing patients: 100%|██████████| 99/99 [00:01<00:00, 54.00it/s]


In [ ]:
# save to pickle
# malignant_proportions_df.to_pickle("data_processed/malignant_proportions_df.pkl")

Plot the spatial arrangement of the cells with assignments.

In [ ]:
from utils.plotting_utils import plot_spatial_arrangement_with_assignment

plot_spatial_arrangement_with_assignment(adata_spatial,
                                         patient_fovs,
                                         legend = True,
                                         save_path = "data_processed/spatial_arrangement_assigned")    

Processing patients: 100%|██████████| 99/99 [00:20<00:00,  4.72it/s]


______

In [ ]:
# NOTE: Here, we can try comptuing the spatial metrics across different radii used for the graph construction.
# We can store each graph separately (small ≈ 30 μm, medium ≈ 100 μm, large ≈ 300 μm). 

In [ ]:
from utils.spatial_utils import analyse_spatial_metrics_multiscale

def process_all_patients_multiple_radii(adata, patient_fovs):
    """
    Process all patients and calculate spatial metrics.

    Parameters:
    -----------------
    adata: AnnData
        The AnnData object containing the data.
    patient_fovs: GroupBy object
        The GroupBy object containing patient id and field of view (FOV) information.
    """
    all_results = []


    for (patient, fov), indices in tqdm(patient_fovs.indices.items(), desc="Processing patients"):
        print(f"Analyzing {patient} - FOV {fov}")
        
        # Subset the data
        adata_subset = adata[indices].copy()
        
        # Calculate metrics
        results = analyse_spatial_metrics_multiscale(adata_subset, patient, fov)
        
        # Store results
        all_results.append(results)

    return all_results

multiple_radii_results = process_all_patients_multiple_radii(adata_spatial, patient_fovs)

In [181]:
# Save the results
# with open("data_processed/multiple_radii_results.json", "w") as file:
#     json.dump(multiple_radii_results, file)

# Load the pre-computed results
with open("data_processed/multiple_radii_results.json", "r") as file:
    multiple_radii_results = json.load(file)

In [182]:
# get the cell types automatically from adata_survival

(patient, fov), idx = next(iter(patient_fovs.groups.items()))
adata_subset = adata_spatial[idx].copy()
cell_types = adata_subset.obs["Cell_type"].cat.categories.tolist()

cell_types = [item for item in cell_types if item not in ["Malignant.Other"]]
cell_types

['B.cell',
 'Endothelial',
 'Fibroblast',
 'Malignant.C10',
 'Malignant.C3',
 'Malignant.C4',
 'Malignant.EMT',
 'Malignant.ciliated',
 'Mast.cell',
 'Monocyte',
 'TNK.cell']

In [ ]:
def prepare_for_survival(multiple_radii_results,
                         cell_types,
                         focus_label="Malignant.EMT",
                         radii=("30um","100um","300um"),
                         partners=None,
                         centrality_exclude=("Malignant.Other",),
                         include_all_centrality=True):
    """
    Build survival feature table (one row per patient,FOV).
    Ripley: only auc_clustering (all cell types).
    Centrality: all cell types (unless excluded) if include_all_centrality=True,
                otherwise only focus_label.
    Neighborhood enrichment + CLQ: focus_label vs partners.
    """
    if partners is None:
        # NOTE: "Malignant.Other" should be already excluded from the cell_types list
        partners = [c for c in cell_types if c not in {focus_label, "Malignant.Other"}]

    rows = []
    for res in multiple_radii_results:
        patient = res.get("patient")
        fov = res.get("FOV")
        row = {"patient": patient, "FOV": fov}

        # Ripley AUC for all cell types
        for ct, feat_dict in res.get("ripley", {}).items():
            row[f"ripley_{ct}_auc_clustering"] = feat_dict.get("auc_clustering", np.nan)

        # Per-radius metrics
        mbr = res.get("metrics_by_radius", {})
        for r in radii:
            r_metrics = mbr.get(r, {})

            # Centrality
            for metric_name, ct_map in r_metrics.get("centrality", {}).items():
                if not isinstance(ct_map, dict):
                    continue
                if include_all_centrality:
                    for ct, val in ct_map.items():
                        if ct in cell_types and ct not in centrality_exclude:
                            row[f"{r}_degree_centrality_{ct}_{metric_name}"] = val
                else:
                    row[f"{r}_centrality_{focus_label}_{metric_name}"] = ct_map.get(focus_label, np.nan)

            # Neighborhood enrichment (z-score) focus_label vs partners
            enrich_z = r_metrics.get("enrichment", {}).get("zscore", {})
            # enrich_z structure: partner -> {focus_label: value} OR focus_label -> {partner: value}
            for partner in partners:
                # Try both orientations safely
                val = np.nan
                if partner in enrich_z and isinstance(enrich_z[partner], dict):
                    val = enrich_z[partner].get(focus_label, val)
                if np.isnan(val) and focus_label in enrich_z and isinstance(enrich_z[focus_label], dict):
                    val = enrich_z[focus_label].get(partner, val)
                row[f"{r}_nhood_enrichment_{focus_label}_{partner}"] = val

            # CLQ focus_label->partner
            clq = r_metrics.get("co_localization", {})
            for partner in partners:
                row[f"{r}_CLQ_{focus_label}_{partner}"] = clq.get(f"{focus_label}_{partner}", np.nan)

        rows.append(row)

    survival_df = (pd.DataFrame(rows)
                     .set_index(["patient","FOV"])
                     .sort_index())

    return survival_df

# Rebuild with all centrality features
survival_data = prepare_for_survival(multiple_radii_results, cell_types)
survival_data.head()

ripley_B.cell_auc_clustering  ripley_Endothelial_auc_clustering  \
patient FOV                                                                     
HGSC1   F001                  66213.459789                       89539.363282   
HGSC10  F018                   5059.460310                       61966.471319   
        F020                  20560.759140                       33359.102662   
HGSC102 F004                           NaN                      106083.251463   
        F005                           NaN                       68659.890679   

              ripley_Fibroblast_auc_clustering  \
patient FOV                                      
HGSC1   F001                      5.597913e+05   
HGSC10  F018                      5.626921e+05   
        F020                      5.433683e+05   
HGSC102 F004                      1.320168e+06   
        F005                      1.215900e+05   

              ripley_Malignant.C10_auc_clustering  \
patient FOV                                         
HGSC1   F001                         1.067450e+06   
HGSC10  F018                         5.732536e+05   
        F020                         1.269598e+06   
HGSC102 F004                         9.150710e+04   
        F005                         6.408253e+05   

              ripley_Malignant.C3_auc_clustering  \
patient FOV                                        
HGSC1   F001                        61806.424385   
HGSC10  F018                       185362.151804   
        F020                        66611.545691   
HGSC102 F004                        13087.317243   
        F005                        12544.964102   

              ripley_Malignant.C4_auc_clustering  \
patient FOV                                        
HGSC1   F001                       240069.399087   
HGSC10  F018                       830181.694594   
        F020                       496697.590995   
HGSC102 F004                       164532.178357   
        F005                        11645.319705   

              ripley_Malignant.EMT_auc_clustering  \
patient FOV                                         
HGSC1   F001                         1.623359e+05   
HGSC10  F018                         2.135655e+05   
        F020                         8.197738e+04   
HGSC102 F004                         9.498546e+05   
        F005                         2.356231e+06   

              ripley_Malignant.Other_auc_clustering  \
patient FOV                                           
HGSC1   F001                          430807.798000   
HGSC10  F018                          316110.142851   
        F020                          330858.261977   
HGSC102 F004                          241573.939257   
        F005                          418897.822243   

              ripley_Malignant.ciliated_auc_clustering  \
patient FOV                                              
HGSC1   F001                             754060.083581   
HGSC10  F018                             482504.349946   
        F020                             476767.905175   
HGSC102 F004                             456165.677226   
        F005                              60276.006938   

              ripley_Mast.cell_auc_clustering  ...  \
patient FOV                                    ...   
HGSC1   F001                      2799.213258  ...   
HGSC10  F018                              NaN  ...   
        F020                              NaN  ...   
HGSC102 F004                      4515.782959  ...   
        F005                              NaN  ...   

              300um_CLQ_Malignant.EMT_B.cell  \
patient FOV                                    
HGSC1   F001                     1063.367415   
HGSC10  F018                             NaN   
        F020                     1162.061517   
HGSC102 F004                             NaN   
        F005                             NaN   

              300um_CLQ_Malignant.EMT_Endothelial  \
patient FOV                                         
HGSC1 

In [ ]:
# Export to csv for further analysis.
# survival_data.to_csv('data_processed/spatial_metrics.csv')